In [25]:
import pandas as pd
import os
import glob

# ==========================================================
# CẤU HÌNH ĐƯỜNG DẪN GÁN CỨNG (Dựa trên cấu trúc folder của bạn)
# ==========================================================
# Thư mục chứa tất cả các file
BASE_FOLDER = r'D:\Github\ai biet\ML-Lab1-Restaurant\label'

# File gốc dùng để đối chiếu
ORIGINAL_DATA_FILE = os.path.join(BASE_FOLDER, 'reviews_final_merged.csv')

# Các file đầu ra
OUTPUT_LABELS_ANSWER = os.path.join(BASE_FOLDER, 'labels_merged_answer.csv') # File đáp án gộp
OUTPUT_WITH_LABEL_3 = os.path.join(BASE_FOLDER, 'reviews_with_label_3.csv')   # File có nhãn 3
OUTPUT_WITHOUT_LABEL_3 = os.path.join(BASE_FOLDER, 'reviews_without_label_3.csv') # File không có nhãn 3
# ==========================================================

def process():
    print("--- 1. Đang gộp 5 file nhãn CSV ---")
    
    # Lấy danh sách tất cả file .csv trong folder
    all_csv_files = glob.glob(os.path.join(BASE_FOLDER, "*.csv"))
    
    # Các file cần loại bỏ (không phải file nhãn thô để gộp)
    exclude_list = [
        'reviews_final_merged.csv',
        'labels_merged_answer.csv',
        'reviews_with_label_3.csv',
        'reviews_without_label_3.csv'
    ]
    
    label_dfs = []
    for f in all_csv_files:
        fname = os.path.basename(f)
        if fname in exclude_list:
            continue
            
        print(f"   + Đang đọc file nhãn: {fname}")
        # Đọc file nhãn không header, gán tạm tên cột là 'id' và 'label'
        df_temp = pd.read_csv(f, header=None, names=['id', 'label'])
        label_dfs.append(df_temp)
    
    if not label_dfs:
        print("Lỗi: Không tìm thấy 5 file nhãn CSV để gộp.")
        return

    # Gộp lại và sắp xếp theo ID (5 chữ số)
    df_all_labels = pd.concat(label_dfs, ignore_index=True)
    df_all_labels['id'] = df_all_labels['id'].astype(int)
    df_all_labels = df_all_labels.sort_values('id').reset_index(drop=True)
    
    # Lưu file đáp án tổng hợp (không header theo định dạng nhãn thô)
    df_all_labels.to_csv(OUTPUT_LABELS_ANSWER, index=False, header=False)
    print(f"-> Đã tạo file đáp án gộp: {OUTPUT_LABELS_ANSWER}")

    print("\n--- 2. Kiểm tra thừa thiếu dòng với file gốc ---")
    if not os.path.exists(ORIGINAL_DATA_FILE):
        print(f"Lỗi: Không tìm thấy file {ORIGINAL_DATA_FILE}")
        return
        
    df_orig = pd.read_csv(ORIGINAL_DATA_FILE)
    id_col = df_orig.columns[0] # Lấy tên cột ID từ file gốc
    
    orig_ids = set(df_orig[id_col].astype(int))
    label_ids = set(df_all_labels['id'])
    
    missing = orig_ids - label_ids
    extra = label_ids - orig_ids
    
    print(f"   + Tổng số dòng file gốc: {len(df_orig)}")
    print(f"   + Tổng số dòng nhãn gộp: {len(df_all_labels)}")
    
    if not missing and not extra:
        print("   => Kết quả: Khớp hoàn toàn, không thừa thiếu dòng.")
    else:
        if missing: print(f"   => Thiếu nhãn cho các ID: {list(missing)[:10]}...")
        if extra: print(f"   => Nhãn thừa cho các ID (không có trong gốc): {list(extra)[:10]}...")

    print("\n--- 3. Xử lý dữ liệu và nhãn 3 ---")
    # Khớp nhãn vào file gốc
    # Đổi tên cột id của nhãn cho trùng với file gốc để merge
    df_all_labels.columns = [id_col, 'label']
    df_final = pd.merge(df_orig, df_all_labels, on=id_col, how='inner')
    
    # Hiển thị các dòng có nhãn 3
    df_3 = df_final[df_final['label'] == 3]
    print(f"-> Tìm thấy {len(df_3)} dòng có nhãn 3. Danh sách ID:")
    print(df_3[id_col].tolist())
    
    # Xuất file có nhãn 3
    df_3.to_csv(OUTPUT_WITH_LABEL_3, index=False, encoding='utf-8-sig')
    
    # Xuất file không có nhãn 3
    df_no_3 = df_final[df_final['label'] != 3]
    df_no_3.to_csv(OUTPUT_WITHOUT_LABEL_3, index=False, encoding='utf-8-sig')

    print(f"\n--- HOÀN TẤT ---")
    print(f"1. File đáp án tổng: {os.path.basename(OUTPUT_LABELS_ANSWER)}")
    print(f"2. File chứa nhãn 3: {os.path.basename(OUTPUT_WITH_LABEL_3)}")
    print(f"3. File sạch (loại bỏ 3): {os.path.basename(OUTPUT_WITHOUT_LABEL_3)}")

if __name__ == "__main__":
    process()

--- 1. Đang gộp 5 file nhãn CSV ---
   + Đang đọc file nhãn: 00001-08000.csv
   + Đang đọc file nhãn: 08001_16000.csv
   + Đang đọc file nhãn: 16001-24000.csv
   + Đang đọc file nhãn: 24001-32000.csv
   + Đang đọc file nhãn: 32001_32807.csv
-> Đã tạo file đáp án gộp: D:\Github\ai biet\ML-Lab1-Restaurant\label\labels_merged_answer.csv

--- 2. Kiểm tra thừa thiếu dòng với file gốc ---
   + Tổng số dòng file gốc: 32806
   + Tổng số dòng nhãn gộp: 32957
   => Nhãn thừa cho các ID (không có trong gốc): [1]...

--- 3. Xử lý dữ liệu và nhãn 3 ---
-> Tìm thấy 94 dòng có nhãn 3. Danh sách ID:
[4033, 10032, 11167, 11966, 11967, 13536, 14116, 14119, 14208, 15491, 15994, 16013, 16071, 16237, 16419, 17049, 17136, 17293, 17294, 17503, 17824, 17832, 17957, 18012, 18071, 18072, 18304, 18674, 18700, 18713, 18716, 18719, 18733, 18966, 19395, 19643, 19858, 19896, 19936, 20574, 21150, 21539, 21610, 21735, 22030, 22082, 22174, 22515, 22634, 22668, 22717, 22729, 22873, 23162, 23478, 23697, 23875, 23891, 240

In [ ]:
import pandas as pd
import os
import glob
from IPython.display import display

# ==========================================================
# 1. CẤU HÌNH ĐƯỜNG DẪN GÁN CỨNG
# ==========================================================
BASE_FOLDER = r'D:\Github\ai biet\ML-Lab1-Restaurant\label'
ORIGINAL_DATA_FILE = os.path.join(BASE_FOLDER, 'reviews_final_merged.csv')

# Các file đầu ra
OUTPUT_DEDUP_LABELS = os.path.join(BASE_FOLDER, 'all_labels_deduplicated.csv')
OUTPUT_WITH_LABEL_3 = os.path.join(BASE_FOLDER, 'reviews_with_label_3.csv')
OUTPUT_WITHOUT_LABEL_3 = os.path.join(BASE_FOLDER, 'reviews_without_label_3.csv')

# ==========================================================
# 2. GỘP NHÃN VÀ KIỂM TRA TRÙNG LẶP (HIỆN RA CELL)
# ==========================================================
print("--- ĐANG THU THẬP DỮ LIỆU NHÃN ---")
all_files = glob.glob(os.path.join(BASE_FOLDER, "*.csv"))
# Loại trừ các file không phải nhãn thô
exclude = ['reviews_final_merged.csv', 'all_labels_deduplicated.csv', 
           'reviews_with_label_3.csv', 'reviews_without_label_3.csv']

list_dfs = []
for f in all_files:
    if os.path.basename(f) in exclude: continue
    df_temp = pd.read_csv(f, header=None, names=['id', 'label'])
    list_dfs.append(df_temp)

df_raw = pd.concat(list_dfs, ignore_index=True)
df_raw['id'] = df_raw['id'].astype(int)
df_raw = df_raw.sort_values('id').reset_index(drop=True)

# --- XUẤT THÔNG TIN TRÙNG RA CELL ---
duplicates = df_raw[df_raw.duplicated(subset=['id'], keep=False)]

if not duplicates.empty:
    print(f"\n[!] PHÁT HIỆN {len(duplicates)} DÒNG BỊ TRÙNG ID. CHI TIẾT BÊN DƯỚI:")
    # Hiển thị bảng trùng lặp ngay tại Cell
    display(duplicates) 
else:
    print("\n[OK] Không có ID nào bị trùng lặp.")

# ==========================================================
# 3. XÓA TRÙNG VÀ XỬ LÝ FILE GỐC
# ==========================================================
# Xóa trùng (giữ lại dòng đầu tiên)
df_label_clean = df_raw.drop_duplicates(subset=['id'], keep='first').reset_index(drop=True)
df_label_clean.to_csv(OUTPUT_DEDUP_LABELS, index=False, encoding='utf-8-sig')

# Đọc file gốc để khớp
df_orig = pd.read_csv(ORIGINAL_DATA_FILE)
id_col_name = df_orig.columns[0] # Lấy tên cột ID từ gốc

# Đồng bộ tên cột ID để merge
df_label_clean.columns = [id_col_name, 'label']
df_merged = pd.merge(df_orig, df_label_clean, on=id_col_name, how='inner')

# Tách 2 file dựa trên nhãn 3
df_with_3 = df_merged[df_merged['label'] == 3]
df_without_3 = df_merged[df_merged['label'] != 3]

# --- THÔNG BÁO KẾT QUẢ ---
print(f"\n--- KẾT QUẢ XỬ LÝ ---")
print(f"- Tổng dòng file gốc: {len(df_orig)}")
print(f"- Tổng dòng nhãn sau khi xóa trùng: {len(df_label_clean)}")
print(f"- Số dòng mang nhãn 3: {len(df_with_3)}")
print(f"- Số dòng đã làm sạch (không nhãn 3): {len(df_without_3)}")

if not df_with_3.empty:
    print(f"- Danh sách ID nhãn 3: {df_with_3[id_col_name].tolist()}")

# Xuất file
df_with_3.to_csv(OUTPUT_WITH_LABEL_3, index=False, encoding='utf-8-sig')
df_without_3.to_csv(OUTPUT_WITHOUT_LABEL_3, index=False, encoding='utf-8-sig')

print(f"\n=> ĐÃ XUẤT XONG 2 FILE: \n   1. {os.path.basename(OUTPUT_WITH_LABEL_3)}\n   2. {os.path.basename(OUTPUT_WITHOUT_LABEL_3)}")

--- ĐANG THU THẬP DỮ LIỆU NHÃN ---


ValueError: invalid literal for int() with base 10: 'review_id'